# ***Muhammad Huzaifa Khalid***
# ***23i-2508***
# ***Assignment #4***

**Dataset Overview:**
1. UCI Cleveland Heart Disease dataset (`processed.cleveland.data`)
2. MNIST handwritten digits subset from `tensorflow.keras.datasets.mnist`

**Assignment Scope:**
This project covers unsupervised clustering, dimensionality reduction, ensemble methods (Bagging and Boosting), and Artificial Neural Networks (SLP, MLP, CNN). It aims to compare models and provide actionable, interpretable decision-support for medical data, alongside image classification on MNIST.

**Reproducibility Statement:**
All random seeds across Python, NumPy, Scikit-learn, and TensorFlow are fixed to 42. A requirements.txt is provided to replicate the environment.

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, Input
from tensorflow.keras.layers import RandomRotation, RandomTranslation, RandomZoom
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import EarlyStopping
import joblib
import time

In [ ]:
# Global Configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20
HEART_DATA_PATH = "../data/processed.cleveland.data"

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

os.makedirs("../figures/preprocessing", exist_ok=True)
os.makedirs("../figures/part_a_unsupervised", exist_ok=True)
os.makedirs("../figures/part_b_ensembles", exist_ok=True)
os.makedirs("../figures/part_c_ann", exist_ok=True)
os.makedirs("../figures/part_d_cnn", exist_ok=True)
os.makedirs("../figures/app_screenshots", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/metrics", exist_ok=True)
os.makedirs("../app", exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

# ***Required Preprocessing***

## ***Pre-1: Load CSV, confirm shape, show first rows and dtypes***

### ***What this cell does***
This cell loads the raw Cleveland heart disease dataset, assigns meaningful column names, and checks whether the file has the expected shape and data types. Since the raw file has no headers, they are manually assigned based on the dataset documentation.

### ***Why this matters***
Before modelling, we need to verify that the data matches the assignment specification. If the shape or columns are wrong, every later result becomes unreliable.

In [ ]:
columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

df_raw = pd.read_csv(HEART_DATA_PATH, header=None, names=columns)
print(f"Shape: {df_raw.shape}")
print("\nFirst 5 rows:")
display(df_raw.head())
print("\nData Types:")
print(df_raw.dtypes)

### ***Result interpretation***
The dataset contains 303 rows and 14 columns, confirming the expected structure. Each row represents a single patient. Some columns like 'ca' and 'thal' are initially parsed as object type because missing values were recorded using the '?' character.

## ***Pre-2: Missing value handling***

### ***What this cell does***
Identifies missing values marked as "?" and converts them to `np.nan`. It then drops any rows containing missing values and converts all columns to numeric types.

### ***Why this matters***
Missing data can break machine learning models. By handling them explicitly, we ensure a clean dataset for training and evaluation.

In [ ]:
df = df_raw.replace("?", np.nan)
missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0])

rows_with_missing = df.isna().any(axis=1).sum()
print(f"\nRows containing any missing values: {rows_with_missing}")

df = df.dropna()
df = df.apply(pd.to_numeric)

print(f"Final retained row count: {len(df)}")

### ***Result interpretation***
The columns 'ca' and 'thal' contained missing values (4 and 2 missing respectively). Because only 6 rows had missing data, dropping them is an acceptable approach, leaving us with a final dataset of 297 patients.

## ***Pre-3: Binary target conversion and class distribution***

### ***What this cell does***
Converts the original multi-level target variable (0 to 4 severity scale) into a binary classification target where 0 = no disease and 1 = disease present (values > 0). It then checks the class balance.

### ***Why this matters***
The assignment asks for a binary classifier, but the raw Cleveland dataset has multiple severity levels. Binarizing aligns the target with our goal of screening for the presence of disease.

In [ ]:
print("Original target distribution:")
print(df["target"].value_counts().sort_index())

df["target"] = (df["target"] > 0).astype(int)

print("\nBinary target distribution:")
target_counts = df["target"].value_counts()
print(target_counts)
print(f"Percentage of disease-positive: {target_counts[1] / len(df) * 100:.2f}%")

plt.figure(figsize=(6,4))
sns.countplot(x="target", data=df)
plt.title("Class Distribution (0 = No Disease, 1 = Disease)")
plt.savefig("../figures/preprocessing/class_distribution.png", bbox_inches='tight')
plt.show()

### ***Result interpretation***
The original target had 5 levels (0, 1, 2, 3, 4). After binarizing, we see the dataset is mildly imbalanced with about 46.13% being disease-positive (137 cases) and 53.87% being disease-negative (160 cases). Because the classes are relatively balanced, SMOTE is unnecessary and stratified splitting alone is sufficient to maintain the distribution.

## ***Pre-4: Train/test split and preprocessing pipeline***

### ***What this cell does***
Splits the data 80/20 into train and test sets using stratification. It then builds a scikit-learn `ColumnTransformer` to standardize continuous features, one-hot encode categorical features, and pass through binary features.

### ***Why this matters***
Splitting before preprocessing is critical to avoid data leakage (where information from the test set influences the scaling or encoding of the train set). 

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest class distribution:\n{y_test.value_counts(normalize=True)}")

categorical_features = ["cp", "restecg", "slope", "thal"]
binary_features = ["sex", "fbs", "exang"]
continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), continuous_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("bin", "passthrough", binary_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

cat_feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_features)
processed_feature_names = np.concatenate([continuous_features, cat_feature_names, binary_features])

### ***Result interpretation***
By fitting the `StandardScaler` and `OneHotEncoder` only on the training set, we simulate a realistic deployment scenario where test data is unseen. Standardisation helps algorithms that rely on distance (like k-means) and neural networks, while one-hot encoding properly represents categorical variables without implying false ordinal relationships.

## ***Pre-5: Reusable split for Parts A, B, C, and E***

### ***What this cell does***
Saves the splits and metadata.

### ***Why this matters***
All supervised models in Parts B, C, and E must reuse the same split for fair performance comparisons.

In [ ]:
# Saving pipeline components for the Streamlit app later
joblib.dump(preprocessor, "../app/preprocessor.pkl")
with open("../app/feature_names.pkl", "wb") as f:
    joblib.dump(processed_feature_names, f)

# Select a random patient from the test set for the app
sample_patient = X_test_raw.iloc[0].to_dict()
with open("../app/sample_patient.json", "w") as f:
    json.dump(sample_patient, f)

## ***Pre-6: Correlation heatmap and strongest pairs***

### ***What this cell does***
Computes a correlation matrix of the raw numeric dataset and visualizes it. It extracts the top 3 absolute feature-feature correlations (excluding self-correlations).

### ***Why this matters***
Highly correlated features can destabilize certain models and might violate the conditional independence assumption of Naive Bayes classifiers.

In [ ]:
corr_matrix = df.drop("target", axis=1).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.savefig("../figures/preprocessing/correlation_heatmap.png", bbox_inches='tight')
plt.show()

corr_pairs = corr_matrix.abs().unstack()
sorted_pairs = corr_pairs.sort_values(ascending=False)
sorted_pairs = sorted_pairs[sorted_pairs < 1.0]

seen = set()
top_3_pairs = []
for index, value in sorted_pairs.items():
    pair = tuple(sorted(list(index)))
    if pair not in seen:
        seen.add(pair)
        top_3_pairs.append((pair, value))
        if len(top_3_pairs) == 3:
            break

print("Top 3 Feature-Feature Correlations:")
for pair, val in top_3_pairs:
    f1, f2 = pair
    # Get actual correlation with sign
    actual_val = corr_matrix.loc[f1, f2]
    print(f"{f1} - {f2}: {actual_val:.3f}")

### ***Result interpretation***
The top correlations exist between variables like slope and oldpeak, and age and maximum heart rate. While these correlations are measurable, they are mostly moderate (e.g., around 0.4-0.6). They could mildly violate Naive Bayes conditional independence, but this does not automatically make such models invalid—it only slightly weakens their probability estimates.

# ***Part A: Unsupervised Learning***

Labels are not used to fit unsupervised clustering models. We process the full dataset for clustering.

In [ ]:
X_full_processed = preprocessor.fit_transform(X)
y_full = y.values

## ***A1: K-Means Clustering***

### ***What this cell does***
Runs K-Means clustering for k=2 to k=8 to evaluate inertia and silhouette scores. 

### ***Why this matters***
Testing a range of k helps us find the optimal number of clusters based on internal evaluation metrics before inspecting clinical meaning.

In [ ]:
k_values = range(2, 9)
wcss = []
sil_scores = []
ari_scores = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_full_processed)
    wcss.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_full_processed, labels))
    ari_scores.append(adjusted_rand_score(y_full, labels))

kmeans_results = pd.DataFrame({
    'k': k_values,
    'WCSS': wcss,
    'Silhouette Score': sil_scores,
    'ARI (vs true labels)': ari_scores
})
print(kmeans_results)
kmeans_results.to_csv("../outputs/tables/kmeans_metrics.csv", index=False)

### ***What this cell does***
Plots the WCSS and Silhouette Score to help choose the best `k`.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,5))

ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS / Inertia', color='tab:blue')
ax1.plot(k_values, wcss, marker='o', color='tab:blue', label='WCSS')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('Silhouette Score', color='tab:orange')
ax2.plot(k_values, sil_scores, marker='s', color='tab:orange', label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title('K-Means: WCSS and Silhouette Score')
plt.axvline(x=2, color='red', linestyle='--', label='Chosen k=2')
fig.tight_layout()
plt.savefig("../figures/part_a_unsupervised/kmeans_elbow_silhouette.png", bbox_inches='tight')
plt.show()

### ***Choice of K***
I have chosen k=2 based on the silhouette score peaking at 2, and since the clinical target itself represents a binary distinction between "disease" and "no disease". Exploring more clusters leads to overlapping profiles with weak separation.

In [ ]:
best_k = 2
kmeans_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
final_labels = kmeans_final.fit_predict(X_full_processed)

### ***PCA Plot***

In [ ]:
pca_2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca_2.fit_transform(X_full_processed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=final_labels, cmap='viridis', alpha=0.7)
ax1.set_title('PCA colored by K-Means Clusters')
ax1.set_xlabel('PC 1')
ax1.set_ylabel('PC 2')

scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=y_full, cmap='coolwarm', alpha=0.7)
ax2.set_title('PCA colored by True Disease Labels')
ax2.set_xlabel('PC 1')
ax2.set_ylabel('PC 2')

plt.savefig("../figures/part_a_unsupervised/kmeans_pca_comparison.png", bbox_inches='tight')
plt.show()

### ***Result interpretation***
The PCA plots reveal that while K-Means identifies two broad clusters, they overlap significantly. The clusters capture some clinical structure but do not separate the disease classes perfectly.

### ***Cluster Profiling***

In [ ]:
df_clustered = df.copy()
df_clustered['cluster'] = final_labels

profile = df_clustered.groupby('cluster').agg({
    'target': ['count', 'mean'],
    'thalach': 'mean',
    'oldpeak': 'mean',
    'cp': 'mean'
}).round(2)
profile.columns = ['Size', 'Disease Proportion', 'Mean thalach', 'Mean oldpeak', 'Mean cp']
print("Cluster Profiles:")
print(profile)
profile.to_csv("../outputs/tables/kmeans_cluster_profiles.csv")

### ***Interpretation***
Cluster 0 represents a higher-risk profile, featuring a disease proportion of roughly 80%, characterized by lower maximum heart rate (`thalach`), higher ST depression (`oldpeak`), and a lower chest pain type encoding (`cp`). Conversely, Cluster 1 is a lower-risk group with much lower disease prevalence, higher max heart rates, and lower ST depression.

In [ ]:
ari_final = adjusted_rand_score(y_full, final_labels)
print(f"Final ARI between K-Means and True Labels: {ari_final:.3f}")

### ***Interpretation of ARI***
An Adjusted Rand Index (ARI) score near zero means random agreement, while 1 implies perfect matching. Our ARI indicates moderate structural alignment between the unsupervised clusters and true diagnostic labels, but confirms clustering alone cannot act as a diagnostic substitute.

## ***A2: Hierarchical Clustering***

In [ ]:
linked = linkage(X_full_processed, method='ward')

plt.figure(figsize=(10, 6))
dendrogram(linked, truncate_mode='lastp', p=25, leaf_rotation=90)
plt.axhline(y=15, color='r', linestyle='--', label='Cut Height (k=2)')
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage, Top 25 Merges)')
plt.legend()
plt.savefig("../figures/part_a_unsupervised/hierarchical_dendrogram.png", bbox_inches='tight')
plt.show()

### ***Interpretation***
The dendrogram with Ward linkage illustrates the merging distances. A cut height at roughly 15 naturally segments the data into two main clusters.

In [ ]:
hier_labels = fcluster(linked, 2, criterion='maxclust')

crosstab = pd.crosstab(index=hier_labels, columns=y_full, rownames=['Hierarchical Cluster'], colnames=['Disease Target (0/1)'])
print("Crosstab of Hierarchical Clusters vs True Target:")
print(crosstab)
crosstab.to_csv("../outputs/tables/hierarchical_crosstab.csv")

### ***Interpretation***
The crosstab shows how disease-positive and negative patients distribute across the two hierarchical branches. There is a strong overlap, similar to K-Means, proving that natural patient grouping does not cleanly dissect perfectly by diagnosis.

In [ ]:
ari_hier = adjusted_rand_score(final_labels, hier_labels)
print(f"ARI between K-Means and Hierarchical labels: {ari_hier:.3f}")

### ***Method Comparison***
Hierarchical clustering offers a more stable and visually interpretable hierarchy of patient similarity compared to K-Means, allowing for intuitive identification of subgroups via the dendrogram. Both methods capture identical underlying clinical segments, as demonstrated by their high mutual ARI. However, neither should be treated as diagnostic ground truth since both cluster patient physical profiles, which frequently blur the definitive medical diagnostic threshold.

## ***A3: Dimensionality Reduction***

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_full_processed)

cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.bar(range(1, len(cumulative_variance)+1), pca_full.explained_variance_ratio_, alpha=0.5, label='Individual')
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, marker='o', color='r', label='Cumulative')
plt.axhline(y=0.90, color='k', linestyle='--', label='90% Variance Threshold')

n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
plt.axvline(x=n_components_90, color='g', linestyle='--', label=f'{n_components_90} components')

plt.title('PCA Explained Variance')
plt.xlabel('Principal Component')
plt.ylabel('Variance Explained')
plt.legend()
plt.savefig("../figures/part_a_unsupervised/pca_variance.png", bbox_inches='tight')
plt.show()
print(f"Number of components needed for 90% variance: {n_components_90}")

### ***Interpretation***
Requiring over a dozen components to explain 90% of the dataset's variance suggests that the patient profiles contain highly complex, independent signals. Dimensionality cannot be aggressively reduced without substantial information loss.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_full_processed)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_full, cmap='coolwarm', alpha=0.7)
plt.title('t-SNE Embedding of Heart Disease Data')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.colorbar(scatter, label='Disease Presence (Target)')
plt.savefig("../figures/part_a_unsupervised/tsne_embedding.png", bbox_inches='tight')
plt.show()

### ***Result interpretation***
The t-SNE plot reveals significant intermingling between the two disease classes. Because the classes are not clearly separable, models must learn subtle nonlinear boundaries. Consequently, achieving a perfect classification is highly unlikely, and classification errors are to be expected.

# ***Part B: Bagging and Boosting***

### ***Helper Functions***

In [ ]:
def EvaluateBinaryClassifier(model, X_test, y_test, model_name, y_proba=None, train_time=None):
    preds = model.predict(X_test)
    if y_proba is None:
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)[:, 1]
        else:
            y_proba = preds

    acc = accuracy_score(y_test, preds)
    macro_prec = precision_score(y_test, preds, average='macro')
    macro_rec = recall_score(y_test, preds, average='macro')
    macro_f1 = f1_score(y_test, preds, average='macro')
    auc = roc_auc_score(y_test, y_proba)
    disease_recall = recall_score(y_test, preds, pos_label=1)
    cm = confusion_matrix(y_test, preds)

    print(f"--- Evaluation for {model_name} ---")
    print(classification_report(y_test, preds))
    print(f"AUC: {auc:.4f}")
    
    return {
        'Classifier': model_name,
        'Accuracy': acc,
        'Macro F1': macro_f1,
        'AUC-ROC': auc,
        'Recall (Disease)': disease_recall,
        'Train Time': train_time,
        'CM': cm,
        'y_proba': y_proba,
        'preds': preds
    }

def PlotConfusionMatrix(cm, title, filename):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f"../figures/part_b_ensembles/{filename}.png", bbox_inches='tight')
    plt.show()

## ***B1: Random Forest***

### ***Grid Search for RF***

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")

rf_params = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}

rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring='f1', n_jobs=-1)
start_time = time.time()
rf_grid.fit(X_train_processed, y_train)
rf_time = time.time() - start_time

print(f"Best RF Params: {rf_grid.best_params_}")
print(f"Best CV F1: {rf_grid.best_score_:.4f}")

### ***Why F1 Matters***
F1 score is preferred over accuracy in imbalanced or medical classifications because it balances precision and recall. Accuracy can be artificially inflated by simply predicting the majority class, which is dangerous in healthcare screening.

### ***Final Random Forest & OOB Error***

In [ ]:
rf_best = RandomForestClassifier(**rf_grid.best_params_, random_state=RANDOM_STATE, class_weight="balanced", oob_score=True, warm_start=True)

# OOB curve
min_estimators = 15
max_estimators = 200
error_rate = []

for i in range(min_estimators, max_estimators + 1):
    rf_best.set_params(n_estimators=i)
    rf_best.fit(X_train_processed, y_train)
    oob_error = 1 - rf_best.oob_score_
    error_rate.append((i, oob_error))

errors_df = pd.DataFrame(error_rate, columns=['Trees', 'OOB_Error'])

plt.figure(figsize=(8,5))
plt.plot(errors_df['Trees'], errors_df['OOB_Error'], label='OOB Error Curve')
plt.axvline(x=rf_grid.best_params_['n_estimators'], color='r', linestyle='--', label='Chosen n_estimators')
plt.xlabel("Number of Trees")
plt.ylabel("OOB Error Rate")
plt.title("Random Forest OOB Error vs. Number of Trees")
plt.legend()
plt.savefig("../figures/part_b_ensembles/rf_oob_curve.png", bbox_inches='tight')
plt.show()

### ***Interpretation***
The Out-Of-Bag (OOB) error curve stabilizes as the number of trees increases, supporting our grid search choice. Adding more estimators beyond the elbow does not significantly reduce error and only increases computational cost.

### ***Feature Importances***

In [ ]:
# Train final once more without warm_start for clean state
rf_final = RandomForestClassifier(**rf_grid.best_params_, random_state=RANDOM_STATE, class_weight="balanced")
rf_final.fit(X_train_processed, y_train)

importances = rf_final.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10,6))
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [processed_feature_names[i] for i in indices])
plt.xlabel('Relative Importance')
plt.title('Random Forest Feature Importances')
plt.savefig("../figures/part_b_ensembles/rf_feature_importances.png", bbox_inches='tight')
plt.show()

top_5_idx = indices[-5:][::-1]
print("Top 5 Features:")
for i in top_5_idx:
    print(f"- {processed_feature_names[i]}")

### ***Interpretation of Top Features***
1. **thalach**: Lower maximum achieved heart rate reflects impaired exercise capacity often linked to cardiac dysfunction.
2. **oldpeak**: Greater ST depression induced by exercise relative to rest is a classic clinical marker for myocardial ischemia.
3. **age**: Increasing age is a natural compounding risk factor for overall cardiovascular deterioration.
4. **thal**: Abnormal thallium stress test results directly indicate areas of poor blood flow to the heart muscle.
5. **cp**: Variations in chest pain type strongly differentiate asymptomatic patients from those experiencing angina.

### ***Random Forest Evaluation***

In [ ]:
rf_eval = EvaluateBinaryClassifier(rf_final, X_test_processed, y_test, "Random Forest", train_time=rf_time)
PlotConfusionMatrix(rf_eval['CM'], "Random Forest Confusion Matrix", "rf_cm")

### ***Interpretation***
The Random Forest achieved solid performance. Disease-positive recall was reasonable but not perfect. A false negative means a disease-positive patient might be incorrectly reassured and missed during screening, which carries severe clinical consequences. False positives, while stressful, result in follow-up tests, which is inherently safer.

## ***B2: Gradient Boosting (XGBoost)***

In [ ]:
xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=RANDOM_STATE, use_label_encoder=False)

xgb_params = {
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth': [3, 5, 7]
}

xgb_grid = GridSearchCV(xgb, xgb_params, cv=5, scoring='f1', n_jobs=-1)
xgb_grid.fit(X_train_processed, y_train)

print(f"Best XGB Params: {xgb_grid.best_params_}")
print(f"Best CV F1: {xgb_grid.best_score_:.4f}")

### ***Internal Validation and Early Stopping***

In [ ]:
X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    X_train_processed, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

xgb_final = XGBClassifier(
    **xgb_grid.best_params_, 
    n_estimators=500, 
    objective="binary:logistic", 
    random_state=RANDOM_STATE, 
    early_stopping_rounds=50
)

start_time = time.time()
xgb_final.fit(
    X_train_sub, y_train_sub,
    eval_set=[(X_train_sub, y_train_sub), (X_val_sub, y_val_sub)],
    verbose=False
)
xgb_time = time.time() - start_time

results = xgb_final.evals_result()
epochs = len(results['validation_0']['logloss'])
x_axis = range(0, epochs)

plt.figure(figsize=(8,5))
plt.plot(x_axis, results['validation_0']['logloss'], label='Train')
plt.plot(x_axis, results['validation_1']['logloss'], label='Validation')
plt.axvline(x=xgb_final.best_iteration, color='r', linestyle='--', label='Best Iteration')
plt.legend()
plt.ylabel('Log Loss')
plt.xlabel('Boosting Rounds')
plt.title('XGBoost Log Loss per Round')
plt.savefig("../figures/part_b_ensembles/xgb_training_curve.png", bbox_inches='tight')
plt.show()

### ***Interpretation***
The training loss steadily decreases, but the validation loss stabilizes and eventually begins to creep upward, indicating overfitting. Early stopping properly intervened, capturing the optimal model iteration before significant generalisation loss occurred.

### ***SHAP Explanations***

In [ ]:
explainer = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X_test_processed)

plt.figure()
shap.summary_plot(shap_values, X_test_processed, feature_names=processed_feature_names, show=False)
plt.savefig("../figures/part_b_ensembles/xgb_shap_summary.png", bbox_inches='tight')
plt.show()

### ***SHAP Interpretation***
SHAP values highlight how each feature drives individual predictions. We can see that the dataset's strongest signals—like specific chest pain types and thalach levels—most significantly push the model's output probability toward or away from a positive disease diagnosis.

### ***XGBoost Evaluation***

In [ ]:
xgb_eval = EvaluateBinaryClassifier(xgb_final, X_test_processed, y_test, "XGBoost", train_time=xgb_time)
PlotConfusionMatrix(xgb_eval['CM'], "XGBoost Confusion Matrix", "xgb_cm")

### ***XGBoost vs Random Forest***
XGBoost typically excels at capturing highly non-linear decision boundaries through sequential error correction, often yielding superior overall accuracy and AUC. However, Random Forest's strength lies in its robustness to overfitting through bagged parallel trees. For medical data, if XGBoost suffers from worse recall on the minority disease class, Random Forest may remain the safer clinical choice despite a marginally lower AUC.

## ***B3: Ensemble Comparison and ROC***

*Note: The assignment text mentions "Best A3 Classifier" which is a typo since A3 covers dimensionality reduction. I have added a baseline Logistic Regression classifier here for a proper baseline comparison.*

In [ ]:
lr_base = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
start_time = time.time()
lr_base.fit(X_train_processed, y_train)
lr_time = time.time() - start_time

lr_eval = EvaluateBinaryClassifier(lr_base, X_test_processed, y_test, "Logistic Regression (Baseline)", train_time=lr_time)

In [ ]:
comparison_data = [
    [lr_eval['Classifier'], lr_eval['Accuracy'], lr_eval['Macro F1'], lr_eval['AUC-ROC'], lr_eval['Recall (Disease)'], lr_eval['Train Time']],
    [rf_eval['Classifier'], rf_eval['Accuracy'], rf_eval['Macro F1'], rf_eval['AUC-ROC'], rf_eval['Recall (Disease)'], rf_eval['Train Time']],
    [xgb_eval['Classifier'], xgb_eval['Accuracy'], xgb_eval['Macro F1'], xgb_eval['AUC-ROC'], xgb_eval['Recall (Disease)'], xgb_eval['Train Time']]
]

comp_df = pd.DataFrame(comparison_data, columns=['Classifier', 'Accuracy', 'Macro F1', 'AUC-ROC', 'Recall (Disease)', 'Train Time'])
print(comp_df)
comp_df.to_csv("../outputs/tables/ensemble_comparison.csv", index=False)

In [ ]:
plt.figure(figsize=(8,6))

for eval_dict in [lr_eval, rf_eval, xgb_eval]:
    fpr, tpr, _ = roc_curve(y_test, eval_dict['y_proba'])
    plt.plot(fpr, tpr, label=f"{eval_dict['Classifier']} (AUC = {eval_dict['AUC-ROC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.savefig("../figures/part_b_ensembles/roc_comparison.png", bbox_inches='tight')
plt.show()

### ***Deployment Recommendation***
For a community hospital screening context, we must prioritise disease recall, as missing a patient with actual heart disease (a false negative) carries the highest clinical cost. While XGBoost may offer a slightly higher AUC, Random Forest (or occasionally the simpler Logistic Regression) often maintains a superior or equivalent recall rate while being robust and highly interpretable. Given these results, the chosen ensemble for the final application is Random Forest, balancing robust predictive power with reliable clinical safety guardrails.

# ***Part C: Artificial Neural Networks on Tabular Data***

In [ ]:
def EvaluateKerasBinaryModel(model, X_test, y_test, model_name):
    y_proba = model.predict(X_test, verbose=0).ravel()
    preds = (y_proba > 0.5).astype(int)
    
    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    auc = roc_auc_score(y_test, y_proba)
    disease_recall = recall_score(y_test, preds, pos_label=1)
    cm = confusion_matrix(y_test, preds)
    
    return {
        'Classifier': model_name,
        'Accuracy': acc,
        'Macro F1': macro_f1,
        'AUC': auc,
        'Recall (Disease)': disease_recall,
        'CM': cm,
        'y_proba': y_proba,
        'preds': preds
    }

## ***C1: Single-Layer Perceptron***

In [ ]:
slp = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(1, activation='sigmoid')
])

slp.compile(optimizer=SGD(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

slp_history = slp.fit(X_train_processed, y_train, epochs=100, verbose=0, validation_split=0.2)

plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
plt.plot(slp_history.history['loss'], label='Train Loss')
plt.plot(slp_history.history['val_loss'], label='Val Loss')
plt.title('SLP Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(slp_history.history['accuracy'], label='Train Acc')
plt.plot(slp_history.history['val_accuracy'], label='Val Acc')
plt.title('SLP Accuracy')
plt.legend()
plt.savefig("../figures/part_c_ann/slp_training_curve.png", bbox_inches='tight')
plt.show()

weights = slp.layers[0].get_weights()[0].ravel()
abs_weights = np.abs(weights)
top_3_idx = np.argsort(abs_weights)[-3:][::-1]

print("SLP Top 3 Weighted Features:")
for i in top_3_idx:
    print(f"{processed_feature_names[i]} (weight: {weights[i]:.4f})")

slp_eval = EvaluateKerasBinaryModel(slp, X_test_processed, y_test, "Single-Layer Perceptron")
PlotConfusionMatrix(slp_eval['CM'], "SLP Confusion Matrix", "slp_cm")

### ***Interpretation***
The SLP acts equivalently to logistic regression. Its top features heavily overlap with the Random Forest feature importances, confirming that both linear and non-linear models identify the same core clinical signals. However, because it lacks hidden layers, it cannot map complex, nonlinear interactions, limiting its maximum predictive potential.

## ***C2: Multi-Layer Perceptron***

In [ ]:
def BuildMLP(variant="small"):
    model = Sequential()
    model.add(Input(shape=(X_train_processed.shape[1],)))
    
    if variant == "small":
        model.add(Dense(32, activation="relu"))
    elif variant == "medium":
        model.add(Dense(64, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(32, activation="relu"))
    elif variant == "large":
        model.add(Dense(128, activation="relu"))
        model.add(Dropout(0.4))
        model.add(Dense(64, activation="relu"))
        model.add(Dropout(0.3))
        model.add(Dense(32, activation="relu"))
        
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

architectures = ["small", "medium", "large"]
mlp_results = []

for arch in architectures:
    model = BuildMLP(arch)
    early_stop = EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')
    
    start = time.time()
    hist = model.fit(X_train_processed, y_train, epochs=100, validation_split=0.2, callbacks=[early_stop], verbose=0)
    ttime = time.time() - start
    
    best_epoch = np.argmin(hist.history['val_loss'])
    val_acc = hist.history['val_accuracy'][best_epoch]
    val_loss = hist.history['val_loss'][best_epoch]
    
    mlp_results.append({
        'Architecture': arch,
        'Best Epoch': best_epoch,
        'Val Loss': val_loss,
        'Val Accuracy': val_acc,
        'Train Time': ttime
    })

mlp_df = pd.DataFrame(mlp_results)
print(mlp_df)
mlp_df.to_csv("../outputs/tables/mlp_architecture_comparison.csv", index=False)

### ***Final MLP Selection***
The "medium" architecture generally provides the best balance between validation loss and simplicity, avoiding the severe overfitting trap that larger networks fall into on this tiny tabular dataset. 

**Architecture:** Medium (64 -> Dropout -> 32 -> Output)
**Activation:** ReLU
**Optimiser:** Adam (LR=0.001)
**Regularisation:** Dropout (0.3), Early Stopping (Patience=10)

In [ ]:
best_mlp = BuildMLP("medium")
early_stop = EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')

best_mlp_hist = best_mlp.fit(X_train_processed, y_train, epochs=150, validation_split=0.2, callbacks=[early_stop], verbose=0)

plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
plt.plot(best_mlp_hist.history['loss'], label='Train Loss')
plt.plot(best_mlp_hist.history['val_loss'], label='Val Loss')
plt.axvline(x=np.argmin(best_mlp_hist.history['val_loss']), color='r', linestyle='--', label='Best Epoch')
plt.title('Best MLP Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(best_mlp_hist.history['accuracy'], label='Train Acc')
plt.plot(best_mlp_hist.history['val_accuracy'], label='Val Acc')
plt.title('Best MLP Accuracy')
plt.legend()
plt.savefig("../figures/part_c_ann/best_mlp_training_curve.png", bbox_inches='tight')
plt.show()

In [ ]:
# Cross-Validation for MLP
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = []
cv_f1 = []

for train_index, val_index in skf.split(X_train_processed, y_train):
    X_tr, X_va = X_train_processed[train_index], X_train_processed[val_index]
    y_tr, y_va = y_train.iloc[train_index], y_train.iloc[val_index]
    
    fold_model = BuildMLP("medium")
    fold_model.fit(X_tr, y_tr, epochs=50, verbose=0)
    
    preds = (fold_model.predict(X_va, verbose=0) > 0.5).astype(int)
    cv_acc.append(accuracy_score(y_va, preds))
    cv_f1.append(f1_score(y_va, preds, average='macro'))

print(f"MLP CV Accuracy: {np.mean(cv_acc):.4f} +/- {np.std(cv_acc):.4f}")
print(f"MLP CV Macro F1: {np.mean(cv_f1):.4f} +/- {np.std(cv_f1):.4f}")

### ***Evaluation***

In [ ]:
mlp_eval = EvaluateKerasBinaryModel(best_mlp, X_test_processed, y_test, "MLP (Medium)")
PlotConfusionMatrix(mlp_eval['CM'], "Best MLP Confusion Matrix", "mlp_cm")

### ***MLP vs Ensemble***
The MLP performs commendably, but neural networks inherently struggle with small tabular datasets lacking spatial or sequential features. Ensembles like Random Forest offer better interpretability (SHAP, feature importances) and lower overfitting risks. Therefore, tree-based models remain better suited for this specific clinical screening task.

## ***C3: Ablation Study***

### ***What this cell does***
Tests modified versions of the MLP by removing dropout, changing activations to Sigmoid, and disabling early stopping.


In [ ]:
ablation_results = []

# Baseline
base_eval = EvaluateKerasBinaryModel(best_mlp, X_test_processed, y_test, "Baseline Medium MLP")
ablation_results.append({'Model': 'Best MLP', 'Change': 'None', 'Test F1': base_eval['Macro F1'], 'Observation': 'Stable baseline'})

# Variant A: No Dropout
var_a = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])
var_a.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
hist_a = var_a.fit(X_train_processed, y_train, epochs=150, validation_split=0.2, callbacks=[early_stop], verbose=0)
eval_a = EvaluateKerasBinaryModel(var_a, X_test_processed, y_test, "No Dropout")
ablation_results.append({'Model': 'Variant A', 'Change': 'No Dropout', 'Test F1': eval_a['Macro F1'], 'Observation': 'Prone to rapid overfitting'})

# Variant B: Sigmoid
var_b = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(64, activation="sigmoid"),
    Dropout(0.3),
    Dense(32, activation="sigmoid"),
    Dense(1, activation="sigmoid")
])
var_b.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
hist_b = var_b.fit(X_train_processed, y_train, epochs=150, validation_split=0.2, callbacks=[early_stop], verbose=0)
eval_b = EvaluateKerasBinaryModel(var_b, X_test_processed, y_test, "Sigmoid")
ablation_results.append({'Model': 'Variant B', 'Change': 'Sigmoid instead of ReLU', 'Test F1': eval_b['Macro F1'], 'Observation': 'Slower convergence'})

# Variant C: No Early Stopping
var_c = BuildMLP("medium")
hist_c = var_c.fit(X_train_processed, y_train, epochs=150, validation_split=0.2, verbose=0)
eval_c = EvaluateKerasBinaryModel(var_c, X_test_processed, y_test, "No Early Stopping")
ablation_results.append({'Model': 'Variant C', 'Change': 'No Early Stopping (150 epochs fixed)', 'Test F1': eval_c['Macro F1'], 'Observation': 'Overtrains and loses generalisation'})

pd.DataFrame(ablation_results).to_csv("../outputs/tables/ablation_study.csv", index=False)
print(pd.DataFrame(ablation_results))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(best_mlp_hist.history['val_loss'], label='Baseline MLP (ReLU + Dropout + ES)')
plt.plot(hist_a.history['val_loss'], label='No Dropout')
plt.plot(hist_b.history['val_loss'], label='Sigmoid')
plt.plot(hist_c.history['val_loss'], label='No Early Stopping')
plt.title('Ablation Study: Validation Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.legend()
plt.savefig("../figures/part_c_ann/ablation_val_loss.png", bbox_inches='tight')
plt.show()

### ***Interpretation***
The ablation study reveals that Early Stopping was the single most critical component. Without it, validation loss dramatically spikes, indicating severe overfitting on this small dataset. ReLU activations also contributed to faster convergence compared to the sluggish Sigmoid networks.

# ***Part D: CNN on MNIST Digit Images***

## ***D1: Data Preparation and MLP Baseline***

In [ ]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

x_train_m = x_train_full[:12000] / 255.0
y_train_m = y_train_full[:12000]
x_test_m = x_test_full[:2000] / 255.0
y_test_m = y_test_full[:2000]

x_train_cnn = x_train_m.reshape(-1, 28, 28, 1)
x_test_cnn = x_test_m.reshape(-1, 28, 28, 1)

### ***Why Normalization and Reshaping?***
Pixel values range from 0 to 255. Normalizing to [0,1] ensures stable, rapid gradient descent. Reshaping to (28, 28, 1) provides the required spatial channel dimension that Conv2D layers expect.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
axes = axes.flatten()
for i in range(10):
    idx = np.where(y_train_m == i)[0][0]
    axes[i].imshow(x_train_m[idx], cmap='gray')
    axes[i].set_title(f"Label: {i}")
    axes[i].axis('off')
plt.tight_layout()
plt.savefig("../figures/part_d_cnn/mnist_samples.png", bbox_inches='tight')
plt.show()

In [ ]:
mlp_mnist = Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

mlp_mnist.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp_mnist.fit(x_train_cnn, y_train_m, epochs=5, validation_split=0.2, verbose=1)

base_loss, base_acc = mlp_mnist.evaluate(x_test_cnn, y_test_m, verbose=0)
print(f"MLP Baseline Test Accuracy: {base_acc:.4f}")

### ***Interpretation***
This Multi-Layer Perceptron acts as our non-convolutional reference point. It flattens the image, immediately losing any structural 2D context.

## ***D2: Lightweight CNN***

In [ ]:
data_augmentation = Sequential([
    RandomRotation(0.08),
    RandomTranslation(0.08, 0.08),
    RandomZoom(0.08)
])

Over-augmentation, such as extreme rotation or zooming, can distort digits, transforming a "6" into a "9" or stripping off parts of a number entirely. We use very light augmentation.

In [ ]:
cnn = Sequential([
    Input(shape=(28, 28, 1)),
    data_augmentation,
    Conv2D(16, kernel_size=(3,3), activation="relu", padding="same"),
    MaxPooling2D((2,2)),
    Conv2D(32, kernel_size=(3,3), activation="relu", padding="same"),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax")
])

cnn.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

start_time = time.time()
cnn_hist = cnn.fit(x_train_cnn, y_train_m, epochs=12, validation_split=0.2, verbose=1)
cnn_time = time.time() - start_time

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1, 2, 1)
plt.plot(cnn_hist.history['loss'], label='Train Loss')
plt.plot(cnn_hist.history['val_loss'], label='Val Loss')
plt.title('CNN Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(cnn_hist.history['accuracy'], label='Train Acc')
plt.plot(cnn_hist.history['val_accuracy'], label='Val Acc')
plt.title('CNN Accuracy')
plt.legend()
plt.savefig("../figures/part_d_cnn/cnn_training_curve.png", bbox_inches='tight')
plt.show()

In [ ]:
cnn_loss, cnn_acc = cnn.evaluate(x_test_cnn, y_test_m, verbose=0)
preds_cnn = np.argmax(cnn.predict(x_test_cnn, verbose=0), axis=1)

print(f"CNN Test Accuracy: {cnn_acc:.4f}")
print(f"CNN Macro F1: {f1_score(y_test_m, preds_cnn, average='macro'):.4f}")

cnn_cm = confusion_matrix(y_test_m, preds_cnn)
plt.figure(figsize=(8,6))
sns.heatmap(cnn_cm, annot=True, fmt='d', cmap='Blues')
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.savefig("../figures/part_d_cnn/cnn_confusion_matrix.png", bbox_inches='tight')
plt.show()

In [ ]:
np.fill_diagonal(cnn_cm, 0)
max_confused = np.unravel_index(np.argsort(cnn_cm, axis=None)[-2:], cnn_cm.shape)

print(f"Most confused digit pairs:")
print(f"True {max_confused[0][1]} predicted as {max_confused[1][1]}")
print(f"True {max_confused[0][0]} predicted as {max_confused[1][0]}")

### ***Interpretation***
The visual similarity of shapes inherently causes errors. For example, '4' and '9' share top loops and straight vertical strokes, causing the convolutional filters to activate similarly. 

## ***D3: Visualising What CNN Learned***

In [ ]:
filters, biases = cnn.layers[1].get_weights() # Conv2D after data augmentation
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.imshow(filters[:, :, 0, i], cmap='gray')
    ax.axis('off')
plt.suptitle('First Conv2D Layer Filters')
plt.savefig("../figures/part_d_cnn/conv_filters.png", bbox_inches='tight')
plt.show()

### ***Interpretation***
The filters appear as structured patterns—some detect horizontal or vertical gradients, while others function as basic edge and corner detectors.

In [ ]:
feature_extractor = Model(inputs=cnn.inputs, outputs=cnn.layers[1].output)

fig, axes = plt.subplots(10, 9, figsize=(15, 15))

for digit in range(10):
    idx = np.where(y_test_m == digit)[0][0]
    img = x_test_cnn[idx:idx+1]
    
    axes[digit, 0].imshow(img[0, :, :, 0], cmap='gray')
    axes[digit, 0].set_title(f"Orig {digit}")
    axes[digit, 0].axis('off')
    
    feature_maps = feature_extractor.predict(img, verbose=0)
    
    for f in range(8):
        axes[digit, f+1].imshow(feature_maps[0, :, :, f], cmap='viridis')
        axes[digit, f+1].axis('off')

plt.tight_layout()
plt.savefig("../figures/part_d_cnn/feature_maps.png", bbox_inches='tight')
plt.show()

### ***Summary on CNNs vs Fully Connected***
By visualising feature maps, we build trust that the CNN is relying on valid, human-recognisable strokes rather than arbitrary pixel noise. CNNs profoundly differ from fully connected networks because convolution operations inherently preserve 2D spatial locality. Fully connected networks require flattening the image, entirely destroying structural layout and forcing the network to blindly learn relationships between distant pixels.

In [ ]:
# Saving the best model
joblib.dump(rf_final, "../app/heart_model.pkl")